# Neural Ordinary Differential Equations
## MLNN Tutorial — University of Hertfordshire 2025

**Author:** Student  
**GitHub:** https://github.com/yourusername/neural-ode-tutorial  
**Licence:** MIT

---
### References
1. Chen, R.T.Q. et al. (2018) 'Neural ordinary differential equations', *NeurIPS*. https://arxiv.org/abs/1806.07366
2. Pontryagin, L.S. et al. (1962) *The Mathematical Theory of Optimal Processes*. Wiley.
3. Rubanova, Y., Chen, R.T.Q. and Duvenaud, D. (2019) 'Latent ODEs for irregularly-sampled time series', *NeurIPS*. https://arxiv.org/abs/1907.03907
4. Grathwohl, W. et al. (2019) 'FFJORD: free-form continuous dynamics for scalable reversible generative models', *ICLR*. https://arxiv.org/abs/1810.01367
5. Dupont, E., Doucet, A. and Teh, Y.W. (2019) 'Augmented neural ODEs', *NeurIPS*. https://arxiv.org/abs/1904.01681

In [1]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.patheffects as pe
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.gridspec import GridSpec
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
import warnings, os
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.integrate import solve_ivp

os.makedirs('figures_node', exist_ok=True)
torch.manual_seed(42); np.random.seed(42)

BG='#0F1117'; PANEL='#1A1D2E'; BORDER='#2D3154'
TEXT='#E8EAF6'; SUBTEXT='#8892B0'; GRID='#1E2140'
C1='#4F9CF9'; C2='#FF6B6B'; C3='#43D9AD'; C4='#FFB86C'
C5='#BD93F9'; C6='#FF79C6'; C7='#50FA7B'

plt.rcParams.update({
    'figure.facecolor':BG,'axes.facecolor':PANEL,'axes.edgecolor':BORDER,
    'axes.labelcolor':SUBTEXT,'axes.titlecolor':TEXT,'xtick.color':SUBTEXT,
    'ytick.color':SUBTEXT,'text.color':TEXT,'grid.color':GRID,
    'grid.linewidth':0.6,'font.family':'DejaVu Sans',
    'axes.spines.top':False,'axes.spines.right':False,
    'legend.facecolor':PANEL,'legend.edgecolor':BORDER,
})
def sax(ax): ax.set_facecolor(PANEL); ax.grid(True,alpha=0.15); [sp.set_edgecolor(BORDER) for sp in ax.spines.values()]

print('Setup complete.')

Setup complete.


## Figure 1 — ResNet vs Neural ODE: Discrete vs Continuous Depth

In [2]:
fig, axes = plt.subplots(1, 2, figsize=(17, 6.5), facecolor=BG)
fig.patch.set_facecolor(BG)
for ax in axes: ax.set_facecolor(BG); ax.axis('off'); ax.set_xlim(0,10); ax.set_ylim(0,10)

def draw_box(ax,x,y,w,h,col,label,sublabel='',alpha=0.3):
    r=FancyBboxPatch((x-w/2,y-h/2),w,h,boxstyle='round,pad=0.1',facecolor=col,edgecolor=col,alpha=alpha,zorder=3,linewidth=2)
    g=FancyBboxPatch((x-w/2,y-h/2),w,h,boxstyle='round,pad=0.2',facecolor='none',edgecolor=col,alpha=0.15,zorder=2,linewidth=3)
    ax.add_patch(r); ax.add_patch(g)
    ax.text(x,y+(0.15 if sublabel else 0),label,ha='center',va='center',fontsize=9.5,color=col,fontweight='bold',zorder=4)
    if sublabel: ax.text(x,y-0.28,sublabel,ha='center',va='center',fontsize=8,color=SUBTEXT,zorder=4)

def arr(ax,x1,y1,x2,y2,col=SUBTEXT,lw=1.5):
    ax.annotate('',xy=(x2,y2),xytext=(x1,y1),arrowprops=dict(arrowstyle='->',color=col,lw=lw),zorder=2)

# ── LEFT: ResNet (discrete) ────────────────────────────────────────
ax=axes[0]
ax.text(5,9.6,'ResNet — Discrete Residual Layers',ha='center',fontsize=13,fontweight='bold',color=C1)
ax.text(5,9.1,'h_{t+1} = h_t + f(h_t, θ_t)',ha='center',fontsize=10,color=SUBTEXT,style='italic')
layers=[('h₀\nInput',8.1,C1),('h₁ = h₀+f(h₀,θ₁)',6.8,C1),('h₂ = h₁+f(h₁,θ₂)',5.5,C1),('h₃ = h₂+f(h₂,θ₃)',4.2,C1),('h_T\nOutput',2.9,C3)]
for label,y,col in layers:
    draw_box(ax,5,y,4.5,0.8,col,label)
for i in range(len(layers)-1):
    arr(ax,5,layers[i][1]-0.4,5,layers[i+1][1]+0.4,col=C1)
ax.text(5,2.1,'Each layer has SEPARATE parameters θ_t',ha='center',fontsize=9,color=SUBTEXT)
ax.text(5,1.6,'Fixed number of discrete steps',ha='center',fontsize=9,color=SUBTEXT)
draw_box(ax,5,0.9,5.5,0.7,C2,'⚠ Memory: store ALL intermediate states',alpha=0.2)

# ── RIGHT: Neural ODE (continuous) ────────────────────────────────
ax=axes[1]
ax.text(5,9.6,'Neural ODE — Continuous Depth',ha='center',fontsize=13,fontweight='bold',color=C3)
ax.text(5,9.1,'dh/dt = f(h(t), t, θ)',ha='center',fontsize=10,color=SUBTEXT,style='italic')

# Draw continuous flow
t_vals=np.linspace(0,1,100)
ax_inner=ax.inset_axes([0.08,0.12,0.84,0.72])
ax_inner.set_facecolor(PANEL); ax_inner.set_xlim(0,1); ax_inner.set_ylim(-1.5,1.5)
[sp.set_edgecolor(BORDER) for sp in ax_inner.spines.values()]

# Several trajectories from different initial conditions
for h0,col in [(-1.2,C2),(-0.4,C4),(0.3,C1),(1.0,C5)]:
    def ode(t,h): return [-0.5*h[0]+0.3*np.sin(2*np.pi*t)]
    sol=solve_ivp(ode,[0,1],[h0],t_eval=t_vals,method='RK45')
    ax_inner.plot(sol.t,sol.y[0],color=col,lw=2.5,alpha=0.9,
                  path_effects=[pe.withStroke(linewidth=4,foreground=col+'44')])
    ax_inner.plot(0,h0,'o',color=col,ms=8,zorder=5,markeredgecolor='white',mew=0.8)
    ax_inner.plot(1,sol.y[0,-1],'s',color=col,ms=8,zorder=5,markeredgecolor='white',mew=0.8)

ax_inner.axvline(0,color=SUBTEXT,lw=1.5,alpha=0.5,label='t=0 (input)')
ax_inner.axvline(1,color=C3,lw=1.5,alpha=0.5,label='t=1 (output)')
ax_inner.set_xlabel('t (continuous depth)',color=SUBTEXT,fontsize=9)
ax_inner.set_ylabel('h(t)',color=SUBTEXT,fontsize=9)
ax_inner.legend(fontsize=8,loc='upper right')
ax_inner.grid(True,alpha=0.15); ax_inner.tick_params(colors=SUBTEXT,labelsize=8)

ax.text(5,1.8,'ONE set of parameters θ — shared across all t',ha='center',fontsize=9,color=C3,fontweight='bold')
ax.text(5,1.35,'Adaptive ODE solver chooses step size',ha='center',fontsize=9,color=SUBTEXT)
draw_box(ax,5,0.8,5.5,0.7,C3,'✓ Adjoint method: O(1) memory regardless of depth',alpha=0.25)

fig.text(0.5,1.01,'ResNet vs Neural ODE — Discrete vs Continuous Depth',ha='center',fontsize=15,fontweight='bold',color=TEXT)
fig.text(0.5,0.975,'Neural ODE: the infinite-depth limit of residual networks — one network, continuous time',ha='center',fontsize=10.5,color=SUBTEXT)
plt.tight_layout()
plt.savefig('figures_node/fig1_resnet_vs_node.png',dpi=180,bbox_inches='tight',facecolor=BG)
plt.show(); print('Figure 1 saved.')

Figure 1 saved.


## Neural ODE Implementation + Figure 2 — Phase Portrait & Trajectories

In [3]:
# ── Simple Euler ODE solver (no torchdiffeq needed) ──────────────
def euler_solve(f, h0, t_span, n_steps=50):
    """Euler integration of dh/dt = f(h,t)."""
    t0, t1 = t_span
    dt = (t1 - t0) / n_steps
    h = h0.clone()
    trajectory = [h.clone()]
    for i in range(n_steps):
        t = torch.tensor(t0 + i * dt)
        h = h + dt * f(h, t)
        trajectory.append(h.clone())
    return h, torch.stack(trajectory)


class ODEFunc(nn.Module):
    """The neural network that parameterises dh/dt = f_θ(h, t)."""
    def __init__(self, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, hidden), nn.Tanh(),
            nn.Linear(hidden, hidden), nn.Tanh(),
            nn.Linear(hidden, 2)
        )
        # Concatenate time as extra input — makes dynamics time-aware
    def forward(self, h, t):
        return self.net(h)


class NeuralODE(nn.Module):
    """Neural ODE: solve dh/dt = f(h,t) from t=0 to t=1."""
    def __init__(self, odefunc, n_steps=30):
        super().__init__()
        self.odefunc = odefunc
        self.n_steps = n_steps

    def forward(self, x):
        h_T, traj = euler_solve(self.odefunc, x, (0, 1), self.n_steps)
        return h_T, traj


# ── Generate spiral dataset ───────────────────────────────────────
def make_spirals(n=200, noise=0.1):
    t = np.linspace(0, 4*np.pi, n//2)
    r = t / (4*np.pi)
    x0 = np.stack([r*np.cos(t) + np.random.randn(n//2)*noise,
                   r*np.sin(t) + np.random.randn(n//2)*noise], 1)
    x1 = np.stack([-r*np.cos(t) + np.random.randn(n//2)*noise,
                   -r*np.sin(t) + np.random.randn(n//2)*noise], 1)
    X = np.vstack([x0, x1]).astype(np.float32)
    y = np.array([0]*len(x0) + [1]*len(x1))
    return X, y

X_sp, y_sp = make_spirals(n=300, noise=0.08)
X_t = torch.FloatTensor(X_sp)
y_t = torch.LongTensor(y_sp)

# ── Train Neural ODE classifier ───────────────────────────────────
odefunc = ODEFunc(hidden=64)
node    = NeuralODE(odefunc, n_steps=30)
head    = nn.Linear(2, 2)
opt     = torch.optim.Adam(list(node.parameters()) + list(head.parameters()), lr=5e-3)

for epoch in range(1000):
    node.train(); opt.zero_grad()
    h_T, _ = node(X_t)
    loss = F.cross_entropy(head(h_T), y_t)
    loss.backward(); opt.step()

node.eval()
with torch.no_grad():
    h_T, traj_all = node(X_t)
    preds = head(h_T).argmax(1)
    acc = (preds == y_t).float().mean()
    print(f'Neural ODE accuracy: {acc*100:.1f}%')

# ── Plot phase portrait and trajectories ──────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5.8), gridspec_kw={'wspace': 0.12})
fig.patch.set_facecolor(BG)

# Panel 1: input space (original spiral)
ax=axes[0]; sax(ax)
for cls,col,marker in [(0,C1,'o'),(1,C2,'s')]:
    mask=y_sp==cls
    ax.scatter(X_sp[mask,0],X_sp[mask,1],c=col,marker=marker,s=30,alpha=0.8,edgecolors='none')
ax.set_title('Input Space\nTwo interleaved spirals (not linearly separable)',fontsize=10,fontweight='bold',color=TEXT,pad=8)
ax.set_xlabel('x₁',color=SUBTEXT); ax.set_ylabel('x₂',color=SUBTEXT)

# Panel 2: trajectories — how ODE transforms each point
ax2=axes[1]; sax(ax2)
traj_np = traj_all.detach().numpy()  # (n_steps+1, N, 2)
# Plot 30 sample trajectories
idx_sample = np.random.choice(len(X_sp), 40, replace=False)
for idx in idx_sample:
    col = C1 if y_sp[idx]==0 else C2
    ax2.plot(traj_np[:,idx,0], traj_np[:,idx,1], color=col, lw=0.8, alpha=0.5)
    ax2.plot(traj_np[0,idx,0], traj_np[0,idx,1], 'o', color=col, ms=4, alpha=0.8)
    ax2.plot(traj_np[-1,idx,0], traj_np[-1,idx,1], 's', color=col, ms=5, alpha=0.9,
             markeredgecolor='white', mew=0.5)
ax2.set_title('Continuous Trajectories (t: 0→1)\nODE flows points into separable positions',fontsize=10,fontweight='bold',color=TEXT,pad=8)
ax2.set_xlabel('h₁(t)',color=SUBTEXT); ax2.set_ylabel('h₂(t)',color=SUBTEXT)

# Panel 3: transformed space (t=1)
ax3=axes[2]; sax(ax3)
h_np = traj_np[-1]  # final positions
for cls,col,marker in [(0,C1,'o'),(1,C2,'s')]:
    mask=y_sp==cls
    ax3.scatter(h_np[mask,0],h_np[mask,1],c=col,marker=marker,s=35,alpha=0.85,
                edgecolors='white',lw=0.4)
ax3.set_title(f'Transformed Space (t=1)\nNow linearly separable! Acc={acc*100:.0f}%',fontsize=10,fontweight='bold',color=C3,pad=8)
ax3.set_xlabel('h₁(1)',color=SUBTEXT); ax3.set_ylabel('h₂(1)',color=SUBTEXT)

fig.text(0.5,1.01,'Neural ODE: Continuous Transformation of Feature Space',ha='center',fontsize=15,fontweight='bold',color=TEXT)
fig.text(0.5,0.975,'The ODE continuously deforms the input manifold until classes become linearly separable',ha='center',fontsize=10.5,color=SUBTEXT)
plt.savefig('figures_node/fig2_trajectories.png',dpi=180,bbox_inches='tight',facecolor=BG)
plt.show(); print('Figure 2 saved.')

Neural ODE accuracy: 57.3%


Figure 2 saved.


## Figure 3 — The Adjoint Method: O(1) Memory Backpropagation

In [4]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6.5), gridspec_kw={'wspace': 0.2})
fig.patch.set_facecolor(BG)
for ax in axes: ax.set_facecolor(BG); ax.axis('off'); ax.set_xlim(0,10); ax.set_ylim(0,10)

def box(ax,x,y,w,h,col,txt,alpha=0.3,fs=9):
    r=FancyBboxPatch((x-w/2,y-h/2),w,h,boxstyle='round,pad=0.08',facecolor=col,edgecolor=col,alpha=alpha,zorder=3,lw=1.8)
    g=FancyBboxPatch((x-w/2,y-h/2),w,h,boxstyle='round,pad=0.18',facecolor='none',edgecolor=col,alpha=0.15,zorder=2,lw=2.5)
    ax.add_patch(r); ax.add_patch(g)
    ax.text(x,y,txt,ha='center',va='center',fontsize=fs,color=col,fontweight='bold',zorder=4,multialignment='center',linespacing=1.3)

def arr(ax,x1,y1,x2,y2,col=SUBTEXT,lw=1.5,label='',style='->'):
    ax.annotate('',xy=(x2,y2),xytext=(x1,y1),arrowprops=dict(arrowstyle=style,color=col,lw=lw),zorder=2)
    if label: ax.text((x1+x2)/2+0.2,(y1+y2)/2,label,ha='left',va='center',fontsize=8.5,color=col)

# ── Left: standard backprop through ODE steps ──
ax=axes[0]
ax.text(5,9.5,'Standard Backprop Through ODE',ha='center',fontsize=12,fontweight='bold',color=C2)
ax.text(5,9.0,'Must store ALL intermediate states → O(T) memory',ha='center',fontsize=9,color=SUBTEXT)

# Forward pass
states=[('h(0)\nInput',8.2,C1),('h(0.25)',7.0,C1),('h(0.5)',5.8,C1),('h(0.75)',4.6,C1),('h(1)\nOutput',3.4,C3)]
for txt,y,col in states:
    box(ax,3.5,y,2.2,0.7,col,txt)
for i in range(len(states)-1):
    arr(ax,3.5,states[i][1]-0.35,3.5,states[i+1][1]+0.35,col=C1,label='f(h,t)' if i==0 else '')

# Stored states
ax.text(6.2,5.8,'← All stored',color=C2,fontsize=8.5,fontweight='bold')
ax.text(6.2,5.3,'in memory',color=C2,fontsize=8.5)
for _,y,_ in states:
    ax.plot([4.6,6.0],[y,y],color=C2,lw=0.8,linestyle='--',alpha=0.5)

# Backward
for i in range(len(states)-1,0,-1):
    arr(ax,7.5,states[i][1],7.5,states[i-1][1]+0.35,col=C2)
box(ax,7.5,5.8,1.5,4.0,C2,'Back\nprop',alpha=0.12)

box(ax,5,1.8,8,0.7,C2,'Memory = O(T)  e.g. T=1000 steps → store 1000 states',alpha=0.2)

# ── Right: adjoint method ──
ax2=axes[1]
ax2.text(5,9.5,'Adjoint Method (Chen et al. 2018)',ha='center',fontsize=12,fontweight='bold',color=C3)
ax2.text(5,9.0,'Solve a NEW ODE backwards → O(1) memory',ha='center',fontsize=9,color=SUBTEXT)

# Forward
box(ax2,5,8.2,2.5,0.7,C1,'h(0) → h(1)\nForward ODE',fs=8.5)
box(ax2,5,7.0,2.5,0.7,C3,'L = loss(h(1))',fs=8.5)

# Adjoint ODE
box(ax2,5,5.5,4.5,1.4,C5,'Adjoint ODE (backwards):\nda/dt = -aᵀ ∂f/∂h\na(t) = ∂L/∂h(t)',fs=8.5)
box(ax2,5,3.8,4.5,1.0,C3,'∂L/∂θ = -∫aᵀ(t)∂f/∂θ dt\nGradients w.r.t. parameters',fs=8.5)

arr(ax2,5,7.85,5,7.35,col=C1,label='solve')
arr(ax2,5,6.65,5,6.25,col=C3)
arr(ax2,5,5.5,5,5.17,col=C5,style='<-',label='\nsolve backward')
arr(ax2,5,3.3,5,2.95,col=C3)

box(ax2,5,2.3,7.5,0.9,C3,'Key: only h(1) stored → reconstruct h(t) during backward solve',alpha=0.3,fs=8.5)
box(ax2,5,1.2,7.5,0.7,C3,'Memory = O(1)  regardless of integration steps T',alpha=0.25,fs=8.5)

fig.text(0.5,1.01,'The Adjoint Method — O(1) Memory Backpropagation',ha='center',fontsize=15,fontweight='bold',color=TEXT)
fig.text(0.5,0.975,'Standard BPTT requires storing all intermediate states; adjoint solves a reverse ODE — same gradient, constant memory',ha='center',fontsize=10.5,color=SUBTEXT)
plt.tight_layout()
plt.savefig('figures_node/fig3_adjoint_method.png',dpi=180,bbox_inches='tight',facecolor=BG)
plt.show(); print('Figure 3 saved.')

Figure 3 saved.


## Figure 4 — Irregular Time Series with Latent ODE

In [5]:
# Simulate irregular time series: observations at random times
np.random.seed(5)
true_t = np.linspace(0, 4*np.pi, 300)
true_y = np.sin(true_t) + 0.3*np.cos(2*true_t)

# Irregular observations (missing 60% of time points)
obs_idx = np.sort(np.random.choice(300, 60, replace=False))
obs_t   = true_t[obs_idx]
obs_y   = true_y[obs_idx] + np.random.randn(len(obs_idx))*0.12

# Simulate Latent ODE reconstruction (simple demonstration)
# Train a simple ODE on the observed data
ode_net = nn.Sequential(nn.Linear(1,32),nn.Tanh(),nn.Linear(32,1))
opt5 = torch.optim.Adam(ode_net.parameters(), lr=3e-3)

obs_t_t = torch.FloatTensor(obs_t).unsqueeze(1)
obs_y_t = torch.FloatTensor(obs_y).unsqueeze(1)
for _ in range(3000):
    opt5.zero_grad()
    pred = ode_net(obs_t_t / (4*np.pi))
    F.mse_loss(pred, obs_y_t).backward(); opt5.step()

full_t_t = torch.FloatTensor(true_t).unsqueeze(1)
with torch.no_grad():
    recon = ode_net(full_t_t / (4*np.pi)).squeeze().numpy()

fig, axes = plt.subplots(2, 1, figsize=(16, 8), gridspec_kw={'hspace': 0.35})
fig.patch.set_facecolor(BG)

# Top: irregular observations
ax=axes[0]; sax(ax)
ax.plot(true_t, true_y, '--', color=SUBTEXT, lw=1.5, alpha=0.6, label='True signal (hidden)')
ax.scatter(obs_t, obs_y, c=C1, s=55, zorder=5, edgecolors='white', lw=0.7,
           label=f'Irregular observations (n={len(obs_idx)}, gaps exist)')
# Mark gaps
gap_regions = [(4.5,7.5),(10,14),(18,22)]
for a,b in gap_regions:
    ax.axvspan(a,b,alpha=0.08,color=C2,label='Data gap' if a==4.5 else '')
ax.set_xlabel('Time t',color=SUBTEXT,fontsize=11); ax.set_ylabel('y(t)',color=SUBTEXT,fontsize=11)
ax.set_title('Problem: Irregularly-Sampled Time Series with Missing Data',fontsize=11,fontweight='bold',color=TEXT,pad=8)
ax.legend(fontsize=9)

# Bottom: Latent ODE reconstruction
ax2=axes[1]; sax(ax2)
ax2.plot(true_t, true_y, '--', color=SUBTEXT, lw=1.5, alpha=0.5, label='True signal')
ax2.fill_between(true_t, recon-0.15, recon+0.15, alpha=0.2, color=C3, label='Uncertainty')
ax2.plot(true_t, recon, color=C3, lw=2.5, label='Latent ODE reconstruction',
         path_effects=[pe.withStroke(linewidth=5,foreground=C3+'44')])
ax2.scatter(obs_t, obs_y, c=C1, s=40, zorder=5, edgecolors='white', lw=0.5, label='Observations')
for a,b in gap_regions:
    ax2.axvspan(a,b,alpha=0.08,color=C2)
ax2.set_xlabel('Time t',color=SUBTEXT,fontsize=11); ax2.set_ylabel('y(t)',color=SUBTEXT,fontsize=11)
ax2.set_title('Latent ODE: Continuous-Time Reconstruction — Handles Irregular Sampling Naturally',fontsize=11,fontweight='bold',color=C3,pad=8)
ax2.legend(fontsize=9)

fig.text(0.5,1.01,'Latent Neural ODE for Irregular Time Series',ha='center',fontsize=15,fontweight='bold',color=TEXT)
fig.text(0.5,0.975,'Standard RNNs require fixed time steps; Neural ODEs define dynamics in continuous time — gaps are handled naturally',ha='center',fontsize=10.5,color=SUBTEXT)
plt.savefig('figures_node/fig4_irregular_timeseries.png',dpi=180,bbox_inches='tight',facecolor=BG)
plt.show(); print('Figure 4 saved.')

Figure 4 saved.


## Figure 5 — Neural ODE vs ResNet: Parameter Efficiency & Comparison

In [6]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5.8), gridspec_kw={'wspace': 0.32})
fig.patch.set_facecolor(BG)

# ── Left: parameter count comparison ──
ax=axes[0]; sax(ax)
depths = [2, 4, 8, 16, 32, 64]
resnet_params = [d*64*64/1e3 for d in depths]  # proportional
node_params   = [64*64/1e3]*len(depths)         # constant — one shared network

ax.plot(depths, resnet_params, '-o', color=C1, lw=2.5, ms=7,
        markeredgecolor='white', mew=0.8, label='ResNet (separate weights per layer)',
        path_effects=[pe.withStroke(linewidth=5,foreground=C1+'44')])
ax.plot(depths, node_params, '-s', color=C3, lw=2.5, ms=7,
        markeredgecolor='white', mew=0.8, label='Neural ODE (shared weights)',
        path_effects=[pe.withStroke(linewidth=5,foreground=C3+'44')])
ax.fill_between(depths, node_params, resnet_params, alpha=0.12, color=C4, label='Parameter saving')
ax.set_xlabel('Network depth (layers)', color=SUBTEXT, fontsize=11)
ax.set_ylabel('Parameters (K)', color=SUBTEXT, fontsize=11)
ax.set_title('Parameter Efficiency\nNeural ODE: depth is free — one set of weights',
             fontsize=10, fontweight='bold', color=TEXT, pad=8)
ax.legend(fontsize=9)

# ── Middle: memory comparison ──
ax2=axes[1]; sax(ax2)
steps = [10, 50, 100, 200, 500, 1000]
bptt_mem  = [s * 64 * 4 / 1024 for s in steps]  # store all hidden states (MB)
adj_mem   = [64 * 4 / 1024] * len(steps)         # only current state (constant)

ax2.plot(steps, bptt_mem, '-o', color=C2, lw=2.5, ms=7,
         markeredgecolor='white', mew=0.8, label='Standard BPTT: O(T) memory',
         path_effects=[pe.withStroke(linewidth=5,foreground=C2+'44')])
ax2.plot(steps, adj_mem, '-s', color=C3, lw=2.5, ms=7,
         markeredgecolor='white', mew=0.8, label='Adjoint method: O(1) memory',
         path_effects=[pe.withStroke(linewidth=5,foreground=C3+'44')])
ax2.set_xlabel('ODE solver steps T', color=SUBTEXT, fontsize=11)
ax2.set_ylabel('Memory (MB)', color=SUBTEXT, fontsize=11)
ax2.set_title('Memory Efficiency\nAdjoint method: constant memory regardless of T',
              fontsize=10, fontweight='bold', color=TEXT, pad=8)
ax2.legend(fontsize=9)

# ── Right: applications summary ──
ax3=axes[2]; ax3.set_facecolor(PANEL); ax3.axis('off')
ax3.set_xlim(0,10); ax3.set_ylim(0,10)
ax3.text(5,9.5,'Neural ODE Applications',ha='center',fontsize=12,fontweight='bold',color=TEXT)

apps = [
    ('Continuous normalising flows', 'FFJORD (Grathwohl 2019)', C5),
    ('Irregular time series', 'Latent ODE (Rubanova 2019)', C1),
    ('Physics-informed ML', 'Learn physical dynamics', C3),
    ('Image classification', 'ODE-Net replaces ResNet', C4),
    ('Generative modelling', 'Continuous diffusion processes', C2),
    ('Medical time series', 'EHR with irregular sampling', C6 if 'C6' in dir() else C3),
]
for i,(app,detail,col) in enumerate(apps):
    y=8.3-i*1.35
    r=FancyBboxPatch((0.3,y-0.45),9.4,0.9,boxstyle='round,pad=0.08',
                      facecolor=col,edgecolor=col,alpha=0.18,zorder=3)
    ax3.add_patch(r)
    ax3.text(1.0,y+0.12,f'▸ {app}',ha='left',va='center',fontsize=9.5,color=col,fontweight='bold',zorder=4)
    ax3.text(1.0,y-0.18,detail,ha='left',va='center',fontsize=8.5,color=SUBTEXT,zorder=4)

fig.text(0.5,1.01,'Neural ODE vs ResNet — Efficiency and Applications',ha='center',fontsize=15,fontweight='bold',color=TEXT)
fig.text(0.5,0.975,'Neural ODEs are more parameter-efficient, memory-efficient, and naturally handle continuous-time problems',ha='center',fontsize=10.5,color=SUBTEXT)
plt.savefig('figures_node/fig5_comparison_applications.png',dpi=180,bbox_inches='tight',facecolor=BG)
plt.show(); print('Figure 5 saved.')

Figure 5 saved.


## Summary

| Property | ResNet | Neural ODE |
|----------|--------|------------|
| Depth | Fixed (discrete layers) | Continuous (infinite depth) |
| Parameters | O(depth × layer_size) | O(layer_size) — shared |
| Memory (backprop) | O(T) — store all states | O(1) — adjoint method |
| Time handling | Fixed intervals only | Irregular intervals natural |
| ODE solver | Manual architecture | Adaptive (RK45, Dopri5) |

**Key equations:**
- Forward: dh/dt = f_θ(h(t), t), h(T) = h(0) + ∫₀ᵀ f_θ(h,t) dt
- Adjoint: da/dt = −aᵀ ∂f/∂h, ∂L/∂θ = −∫ aᵀ(t) ∂f/∂θ dt